# 06 - PatchTST (Mirrored 6 Strategies)

This notebook mirrors the 6-strategy federated transfer-learning setup from MLP/LSTM notebooks using a PatchTST-style architecture.


In [ ]:
from pathlib import Path
RUN_MODE = "train"  # "train" or "load"
assert RUN_MODE in {"train", "load"}
MODELS_DIR = Path("saved_models")
RESULTS_DIR = Path("saved_results")
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
print("RUN_MODE:", RUN_MODE)


In [ ]:
import os, random, math, gc, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


In [ ]:
TRAIN_PATH = "ashrae/train.cleaned_isamu_matt.building_mean_y_ge_1.csv"
BMETA_PATH = "ashrae/building_metadata.csv"
WTRAIN_PATH = "ashrae/weather_train.cleaned.csv"

train = pd.read_csv(TRAIN_PATH)
bmeta = pd.read_csv(BMETA_PATH)
wtrain = pd.read_csv(WTRAIN_PATH)

train = train[train["meter"] == 0].copy()
train["timestamp"] = pd.to_datetime(train["timestamp"])
wtrain["timestamp"] = pd.to_datetime(wtrain["timestamp"])

keep_bmeta = [c for c in ["building_id", "site_id", "primary_use", "square_feet", "year_built", "floor_count"] if c in bmeta.columns]
bmeta2 = bmeta[keep_bmeta].copy()

df = train.merge(bmeta2, on="building_id", how="left")
df = df.merge(wtrain, on=["site_id", "timestamp"], how="left")

def add_time_features(d):
    d["hour"] = d["timestamp"].dt.hour
    d["dow"] = d["timestamp"].dt.dayofweek
    d["month"] = d["timestamp"].dt.month
    d["is_weekend"] = (d["dow"] >= 5).astype(int)
    return d

df = add_time_features(df)

weather_cols = [
    "air_temperature", "dew_temperature", "wind_speed", "cloud_coverage",
    "precip_depth_1_hr", "sea_level_pressure"
]
for c in weather_cols:
    if c in df.columns:
        df[c] = df[c].fillna(df[c].median())

for lag in [1, 24]:
    df[f"lag_{lag}"] = df.groupby("building_id")["meter_reading"].shift(lag)

df["roll24_mean"] = df.groupby("building_id")["meter_reading"].shift(1).rolling(24).mean()
df["roll24_std"] = df.groupby("building_id")["meter_reading"].shift(1).rolling(24).std()

df = df[df[["lag_1", "lag_24", "roll24_mean", "roll24_std"]].notna().all(axis=1)].copy()
df["y"] = np.log1p(df["meter_reading"].values)
df.reset_index(drop=True, inplace=True)

FEATURES = ["hour", "dow", "month", "is_weekend", "meter_reading", "lag_1", "lag_24", "roll24_mean", "roll24_std"]
for c in weather_cols:
    if c in df.columns:
        FEATURES.append(c)
FEATURES = [c for c in FEATURES if c in df.columns]
TARGET = "y"
print("Rows:", len(df), "Buildings:", df["building_id"].nunique(), "Features:", len(FEATURES))


In [ ]:
# Experiment config (mirrors MLP/LSTM setting style)
N_CLIENTS = 60
MAX_ROWS_PER_CLIENT = 8000
NON_IID_BY_SITE = True
SITES_TO_USE = 6

TRAIN_FRAC = 0.75
VAL_FRAC = 0.10
ENABLE_COLD_START = True
COLD_START_RATIO = 0.20

ROUNDS = 10
SAMPLE_FRAC = 0.25
LOCAL_EPOCHS = 1
LR = 1e-3

LOOKBACK = 96
HORIZON = 1
BATCH_SIZE = 256

PATCH_LEN = 16
PATCH_STRIDE = 8
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 3
FF_DIM = 256
DROPOUT = 0.1

WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

PATCHTST_LOSS_CURVES = []
CURRENT_PATCHTST_STRATEGY = "PatchTST"

def set_patchtst_curve_strategy(name, reset=False):
    global CURRENT_PATCHTST_STRATEGY, PATCHTST_LOSS_CURVES
    CURRENT_PATCHTST_STRATEGY = str(name)
    if reset:
        PATCHTST_LOSS_CURVES = []


In [ ]:
# client selection + split
all_bids = sorted(df["building_id"].unique())
if NON_IID_BY_SITE and "site_id" in df.columns:
    site_counts = df.groupby("site_id")["building_id"].nunique().sort_values(ascending=False)
    selected_sites = site_counts.index[:SITES_TO_USE].tolist()
    candidate_bids = sorted(df[df["site_id"].isin(selected_sites)]["building_id"].unique())
else:
    candidate_bids = all_bids

rng = np.random.default_rng(SEED)
if len(candidate_bids) > N_CLIENTS:
    usable_bids = sorted(rng.choice(candidate_bids, size=N_CLIENTS, replace=False).tolist())
else:
    usable_bids = candidate_bids

cold_bids = set()
if ENABLE_COLD_START:
    k = max(1, int(len(usable_bids) * COLD_START_RATIO))
    cold_bids = set(rng.choice(usable_bids, size=k, replace=False).tolist())

source_bids = [b for b in usable_bids if b not in cold_bids]

client_data = {}
for bid in usable_bids:
    d = df[df["building_id"] == bid].sort_values("timestamp").copy()
    if len(d) > MAX_ROWS_PER_CLIENT:
        d = d.iloc[-MAX_ROWS_PER_CLIENT:]
    n = len(d)
    i1 = int(n * TRAIN_FRAC)
    i2 = int(n * (TRAIN_FRAC + VAL_FRAC))
    if i1 <= LOOKBACK + 1 or i2 <= i1 + 1 or n <= i2 + 1:
        continue
    client_data[bid] = {"full": d.reset_index(drop=True), "train_end": i1, "val_end": i2, "cold_start": bid in cold_bids}

usable_bids = sorted(client_data.keys())
source_bids = [b for b in usable_bids if b not in cold_bids]
print("usable", len(usable_bids), "source", len(source_bids), "cold", len(cold_bids))


In [ ]:
# scale on source train only
scaler = StandardScaler()
parts = []
for bid in source_bids:
    d = client_data[bid]["full"]
    parts.append(d.iloc[:client_data[bid]["train_end"]][FEATURES])
scaler.fit(pd.concat(parts, axis=0).values)

def make_windows(arr_x, arr_y, start_idx, end_idx, lookback=96, horizon=1):
    xs, ys = [], []
    for t in range(max(start_idx, lookback), end_idx - horizon + 1):
        xs.append(arr_x[t - lookback:t])
        ys.append(arr_y[t + horizon - 1])
    if len(xs) == 0:
        return np.empty((0, lookback, arr_x.shape[1]), np.float32), np.empty((0, 1), np.float32)
    return np.asarray(xs, np.float32), np.asarray(ys, np.float32).reshape(-1, 1)

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def split_windows_for_bid(bid):
    d = client_data[bid]["full"]
    Xall = scaler.transform(d[FEATURES].values).astype(np.float32)
    yall = d[TARGET].values.astype(np.float32)
    tr_end = client_data[bid]["train_end"]
    va_end = client_data[bid]["val_end"]
    Xtr, ytr = make_windows(Xall, yall, 0, tr_end, LOOKBACK, HORIZON)
    Xva, yva = make_windows(Xall, yall, tr_end, va_end, LOOKBACK, HORIZON)
    Xte, yte = make_windows(Xall, yall, va_end, len(d), LOOKBACK, HORIZON)
    return Xtr, ytr, Xva, yva, Xte, yte

def loaders_for_bid(bid, batch=BATCH_SIZE):
    Xtr, ytr, Xva, yva, Xte, yte = split_windows_for_bid(bid)
    tr = DataLoader(SeqDataset(Xtr, ytr), batch_size=batch, shuffle=True)
    va = DataLoader(SeqDataset(Xva, yva), batch_size=batch, shuffle=False)
    te = DataLoader(SeqDataset(Xte, yte), batch_size=batch, shuffle=False)
    return tr, va, te


In [ ]:
# PatchTST-style model with separate encoder/head for personalized and progressive strategies
class PatchTSTEncoder(nn.Module):
    def __init__(self, n_features, lookback, patch_len, patch_stride, d_model, n_heads, n_layers, ff_dim, dropout):
        super().__init__()
        self.patch_len = patch_len
        self.patch_stride = patch_stride
        self.n_patches = 1 + (lookback - patch_len) // patch_stride
        self.patch_proj = nn.Linear(patch_len * n_features, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

    def forward(self, x):
        p = x.unfold(1, self.patch_len, self.patch_stride).permute(0,1,3,2).contiguous()
        p = p.view(x.size(0), self.n_patches, -1)
        z = self.patch_proj(p) + self.pos_emb
        z = self.encoder(z)
        return z.mean(dim=1)

class PatchTSTModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.encoder = PatchTSTEncoder(
            n_features=n_features, lookback=LOOKBACK, patch_len=PATCH_LEN, patch_stride=PATCH_STRIDE,
            d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, ff_dim=FF_DIM, dropout=DROPOUT
        )
        self.head = nn.Sequential(nn.LayerNorm(D_MODEL), nn.Linear(D_MODEL, 1))
    def forward(self, x):
        return self.head(self.encoder(x))

def make_model():
    return PatchTSTModel(len(FEATURES)).to(DEVICE)

def get_weights(model):
    return [v.detach().cpu().numpy() for _, v in model.state_dict().items()]

def set_weights(model, weights):
    sd = model.state_dict()
    for (k, _), w in zip(sd.items(), weights):
        sd[k] = torch.tensor(w)
    model.load_state_dict(sd)

def average_weights(weights_list):
    out = []
    for ws in zip(*weights_list):
        out.append(np.mean(np.stack(ws, axis=0), axis=0))
    return out


In [ ]:
def mean_loader_loss(model, loader, loss_fn):
    model.eval(); tot=0.0; n=0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            loss = loss_fn(model(X), y)
            b = len(y); tot += loss.item() * b; n += b
    return tot / max(n, 1)


def train_one(model, loader, epochs=1, lr=1e-3, val_loader=None, freeze_encoder=False, record_name='PatchTST'):
    if freeze_encoder:
        for p in model.encoder.parameters():
            p.requires_grad = False
    else:
        for p in model.encoder.parameters():
            p.requires_grad = True

    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    hist = {'train_loss': [], 'val_loss': []}

    for _ in range(epochs):
        model.train(); tot=0.0; n=0
        for X,y in loader:
            X,y = X.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP_NORM)
            opt.step()
            b = len(y); tot += loss.item() * b; n += b
        hist['train_loss'].append(tot / max(n,1))
        if val_loader is not None:
            hist['val_loss'].append(mean_loader_loss(model, val_loader, loss_fn))
        else:
            hist['val_loss'].append(np.nan)

    name = str(record_name) if record_name not in [None, '', 'PatchTST'] else str(CURRENT_PATCHTST_STRATEGY)
    PATCHTST_LOSS_CURVES.append({'strategy': name, 'name': name, 'model': 'PatchTST', 'train_loss': hist['train_loss'], 'val_loss': hist['val_loss']})
    return hist

def eval_model(model, loader):
    ys, ps = [], []
    model.eval()
    with torch.no_grad():
        for X, y in loader:
            X = X.to(DEVICE)
            pred = model(X).cpu().numpy().reshape(-1)
            ys.append(y.numpy().reshape(-1)); ps.append(pred)
    if len(ys)==0:
        return np.nan, np.nan, np.nan, np.nan
    y_true = np.concatenate(ys); y_pred = np.concatenate(ps)
    y_true_raw = np.expm1(y_true); y_pred_raw = np.expm1(y_pred)
    mae = mean_absolute_error(y_true_raw, y_pred_raw)
    rmse = math.sqrt(mean_squared_error(y_true_raw, y_pred_raw))
    cvrmse = (rmse / max(np.mean(y_true_raw), 1e-8)) * 100
    wape = (np.sum(np.abs(y_true_raw-y_pred_raw)) / max(np.sum(np.abs(y_true_raw)), 1e-8)) * 100
    return mae, rmse, cvrmse, wape


In [ ]:
# Similarity profiles for instance/similarity-aware strategies
profiles = {}
for bid in source_bids:
    d = client_data[bid]['full'].iloc[:client_data[bid]['train_end']]
    h = d.groupby('hour')['meter_reading'].mean().reindex(range(24), fill_value=0).values
    w = d.groupby('dow')['meter_reading'].mean().reindex(range(7), fill_value=0).values
    profiles[bid] = np.concatenate([h, w])

profile_mat = np.stack([profiles[b] for b in source_bids], axis=0)
profile_nn = NearestNeighbors(n_neighbors=min(5, len(source_bids)), metric='euclidean').fit(profile_mat)

sim_matrix = {}
for i, bi in enumerate(source_bids):
    vi = profiles[bi]
    sims = {}
    for bj in source_bids:
        vj = profiles[bj]
        dist = np.linalg.norm(vi - vj)
        sims[bj] = 1.0 / (1.0 + dist)
    total = sum(sims.values())
    sim_matrix[bi] = {k: v/total for k,v in sims.items()}


In [ ]:
# Pretrain base model on pooled source data
def pooled_loader_for_source(batch=BATCH_SIZE):
    Xs, ys = [], []
    for bid in source_bids:
        Xtr, ytr, _, _, _, _ = split_windows_for_bid(bid)
        if len(Xtr)>0:
            Xs.append(Xtr); ys.append(ytr)
    X = np.concatenate(Xs, axis=0); y = np.concatenate(ys, axis=0)
    return DataLoader(SeqDataset(X,y), batch_size=batch, shuffle=True)

if RUN_MODE == 'train':
    pretrain_model = make_model()
    set_patchtst_curve_strategy('Pretrain-PatchTST', reset=True)
    train_one(pretrain_model, pooled_loader_for_source(), epochs=5, lr=LR, record_name='Pretrain-PatchTST')
    pretrain_weights = get_weights(pretrain_model)
    torch.save(pretrain_model.state_dict(), MODELS_DIR / 'patchtst_pretrain.pt')
else:
    pretrain_model = make_model()
    pretrain_model.load_state_dict(torch.load(MODELS_DIR / 'patchtst_pretrain.pt', map_location=DEVICE))
    pretrain_weights = get_weights(pretrain_model)


In [ ]:
# Strategy 1: FTL
set_patchtst_curve_strategy('FTL', reset=False)
weights = pretrain_weights

if RUN_MODE == 'train':
    for r in range(ROUNDS):
        k = max(2, int(len(source_bids)*SAMPLE_FRAC))
        sampled = random.sample(source_bids, k)
        local_ws = []
        for bid in sampled:
            tr, va, _ = loaders_for_bid(bid)
            m = make_model(); set_weights(m, weights)
            train_one(m, tr, epochs=LOCAL_EPOCHS, lr=LR, val_loader=va)
            local_ws.append(get_weights(m))
        weights = average_weights(local_ws)
    ftl_weights = weights
    with open(MODELS_DIR / 'patchtst_ftl.pkl','wb') as f: pickle.dump(ftl_weights,f)
else:
    with open(MODELS_DIR / 'patchtst_ftl.pkl','rb') as f: ftl_weights = pickle.load(f)


In [ ]:
# Strategy 2: Personalized-FL (global encoder + local heads)
set_patchtst_curve_strategy('Personalized-FL', reset=False)

def split_encoder_head(weights):
    m = make_model(); set_weights(m, weights)
    enc = [v.detach().cpu().numpy() for _, v in m.encoder.state_dict().items()]
    head = [v.detach().cpu().numpy() for _, v in m.head.state_dict().items()]
    return enc, head

def merge_encoder_head(enc_w, head_w):
    m = make_model()
    enc_sd = m.encoder.state_dict(); head_sd = m.head.state_dict()
    for (k,_),w in zip(enc_sd.items(), enc_w): enc_sd[k] = torch.tensor(w)
    for (k,_),w in zip(head_sd.items(), head_w): head_sd[k] = torch.tensor(w)
    m.encoder.load_state_dict(enc_sd); m.head.load_state_dict(head_sd)
    return get_weights(m)

global_enc, base_head = split_encoder_head(pretrain_weights)
pers_heads = {bid: base_head for bid in source_bids}

if RUN_MODE == 'train':
    for r in range(ROUNDS):
        k = max(2, int(len(source_bids)*SAMPLE_FRAC))
        sampled = random.sample(source_bids, k)
        enc_updates = []
        for bid in sampled:
            tr, va, _ = loaders_for_bid(bid)
            m = make_model(); set_weights(m, merge_encoder_head(global_enc, pers_heads[bid]))
            train_one(m, tr, epochs=LOCAL_EPOCHS, lr=LR, val_loader=va)
            e, h = split_encoder_head(get_weights(m))
            enc_updates.append(e); pers_heads[bid] = h
        global_enc = average_weights(enc_updates)
    with open(MODELS_DIR/'patchtst_personalized.pkl','wb') as f: pickle.dump({'enc':global_enc,'heads':pers_heads},f)
else:
    with open(MODELS_DIR/'patchtst_personalized.pkl','rb') as f:
        tmp = pickle.load(f); global_enc, pers_heads = tmp['enc'], tmp['heads']


In [ ]:
# Strategy 3: Progressive Unfreezing
set_patchtst_curve_strategy('Progressive Unfreezing', reset=False)
weights_prog = pretrain_weights

if RUN_MODE == 'train':
    frozen_rounds = max(1, ROUNDS // 3)
    for r in range(ROUNDS):
        freeze_encoder = (r < frozen_rounds)
        k = max(2, int(len(source_bids)*SAMPLE_FRAC))
        sampled = random.sample(source_bids, k)
        local_ws = []
        for bid in sampled:
            tr, va, _ = loaders_for_bid(bid)
            m = make_model(); set_weights(m, weights_prog)
            train_one(m, tr, epochs=LOCAL_EPOCHS, lr=LR, val_loader=va, freeze_encoder=freeze_encoder)
            local_ws.append(get_weights(m))
        weights_prog = average_weights(local_ws)
    with open(MODELS_DIR/'patchtst_progressive.pkl','wb') as f: pickle.dump(weights_prog,f)
else:
    with open(MODELS_DIR/'patchtst_progressive.pkl','rb') as f: weights_prog = pickle.load(f)


In [ ]:
# Strategy 4: Instance-TL (augment each client with nearest source profiles)
set_patchtst_curve_strategy('Instance-TL', reset=False)
weights_inst = pretrain_weights

def augmented_loader_for_bid(bid, k_neighbors=3, batch=BATCH_SIZE):
    Xtr, ytr, _, _, _, _ = split_windows_for_bid(bid)
    if bid in profiles:
        v = profiles[bid].reshape(1,-1)
    else:
        d = client_data[bid]['full'].iloc[:client_data[bid]['train_end']]
        h = d.groupby('hour')['meter_reading'].mean().reindex(range(24), fill_value=0).values
        w = d.groupby('dow')['meter_reading'].mean().reindex(range(7), fill_value=0).values
        v = np.concatenate([h,w]).reshape(1,-1)
    nn_idx = profile_nn.kneighbors(v, return_distance=False)[0]
    nbrs = [source_bids[i] for i in nn_idx[:k_neighbors]]
    Xs, ys = [Xtr], [ytr]
    for nb in nbrs:
        Xn, yn, _, _, _, _ = split_windows_for_bid(nb)
        take = min(len(Xn), max(50, len(Xtr)//4))
        if take > 0:
            Xs.append(Xn[:take]); ys.append(yn[:take])
    X = np.concatenate(Xs, axis=0); y = np.concatenate(ys, axis=0)
    return DataLoader(SeqDataset(X,y), batch_size=batch, shuffle=True)

if RUN_MODE == 'train':
    for r in range(ROUNDS):
        k = max(2, int(len(source_bids)*SAMPLE_FRAC))
        sampled = random.sample(source_bids, k)
        local_ws = []
        for bid in sampled:
            tr_aug = augmented_loader_for_bid(bid)
            _, va, _ = loaders_for_bid(bid)
            m = make_model(); set_weights(m, weights_inst)
            train_one(m, tr_aug, epochs=LOCAL_EPOCHS, lr=LR, val_loader=va)
            local_ws.append(get_weights(m))
        weights_inst = average_weights(local_ws)
    with open(MODELS_DIR/'patchtst_instance.pkl','wb') as f: pickle.dump(weights_inst,f)
else:
    with open(MODELS_DIR/'patchtst_instance.pkl','rb') as f: weights_inst = pickle.load(f)


In [ ]:
# Strategy 5: Fed-SimTL (similarity-weighted aggregation)
set_patchtst_curve_strategy('Fed-SimTL', reset=False)
weights_sim = pretrain_weights

def weighted_average_weights(weights_list, coeffs):
    coeffs = np.asarray(coeffs, dtype=np.float64)
    coeffs = coeffs / max(coeffs.sum(), 1e-12)
    out = []
    for ws in zip(*weights_list):
        arr = np.stack(ws, axis=0)
        out.append(np.tensordot(coeffs, arr, axes=(0,0)))
    return out

if RUN_MODE == 'train':
    for r in range(ROUNDS):
        k = max(2, int(len(source_bids)*SAMPLE_FRAC))
        sampled = random.sample(source_bids, k)
        local_ws, local_sc = [], []
        for bid in sampled:
            tr, va, _ = loaders_for_bid(bid)
            m = make_model(); set_weights(m, weights_sim)
            train_one(m, tr, epochs=LOCAL_EPOCHS, lr=LR, val_loader=va)
            local_ws.append(get_weights(m))
            local_sc.append(sim_matrix[bid][bid])
        weights_sim = weighted_average_weights(local_ws, local_sc)
    with open(MODELS_DIR/'patchtst_simtl.pkl','wb') as f: pickle.dump(weights_sim,f)
else:
    with open(MODELS_DIR/'patchtst_simtl.pkl','rb') as f: weights_sim = pickle.load(f)


In [ ]:
# Strategy 6: FedMetaTL (Reptile-style meta init + FL)
set_patchtst_curve_strategy('FedMetaTL', reset=False)

if RUN_MODE == 'train':
    meta_weights = pretrain_weights
    META_ROUNDS = 5
    META_STEP = 0.2
    for _ in range(META_ROUNDS):
        k = max(2, int(len(source_bids)*SAMPLE_FRAC))
        sampled = random.sample(source_bids, k)
        locals_w = []
        for bid in sampled:
            tr, va, _ = loaders_for_bid(bid)
            m = make_model(); set_weights(m, meta_weights)
            train_one(m, tr, epochs=2, lr=LR, val_loader=va, record_name='meta-inner')
            locals_w.append(get_weights(m))
        mean_w = average_weights(locals_w)
        meta_weights = [w + META_STEP * (mw - w) for w, mw in zip(meta_weights, mean_w)]

    weights_meta = meta_weights
    for r in range(ROUNDS):
        k = max(2, int(len(source_bids)*SAMPLE_FRAC))
        sampled = random.sample(source_bids, k)
        local_ws = []
        for bid in sampled:
            tr, va, _ = loaders_for_bid(bid)
            m = make_model(); set_weights(m, weights_meta)
            train_one(m, tr, epochs=LOCAL_EPOCHS, lr=LR, val_loader=va)
            local_ws.append(get_weights(m))
        weights_meta = average_weights(local_ws)
    with open(MODELS_DIR/'patchtst_fedmeta.pkl','wb') as f: pickle.dump(weights_meta,f)
else:
    with open(MODELS_DIR/'patchtst_fedmeta.pkl','rb') as f: weights_meta = pickle.load(f)


In [ ]:
# Baseline 0: Local-only and Baseline C: Centralized Global
set_patchtst_curve_strategy('0. Local-only', reset=False)

if RUN_MODE == 'train':
    local_only_weights = {}
    local_epochs = max(1, ROUNDS * LOCAL_EPOCHS)
    for bid in usable_bids:
        tr, va, _ = loaders_for_bid(bid)
        m = make_model()
        train_one(m, tr, epochs=local_epochs, lr=LR, val_loader=va, record_name='0. Local-only')
        local_only_weights[bid] = get_weights(m)
    with open(MODELS_DIR/'patchtst_local_only.pkl','wb') as f: pickle.dump(local_only_weights,f)
else:
    with open(MODELS_DIR/'patchtst_local_only.pkl','rb') as f: local_only_weights = pickle.load(f)

set_patchtst_curve_strategy('C. Centralized', reset=False)

def pooled_loader_for_all(batch=BATCH_SIZE):
    Xs, ys = [], []
    for bid in usable_bids:
        Xtr, ytr, _, _, _, _ = split_windows_for_bid(bid)
        if len(Xtr) > 0:
            Xs.append(Xtr); ys.append(ytr)
    X = np.concatenate(Xs, axis=0); y = np.concatenate(ys, axis=0)
    return DataLoader(SeqDataset(X, y), batch_size=batch, shuffle=True)

if RUN_MODE == 'train':
    central_model = make_model()
    train_one(central_model, pooled_loader_for_all(), epochs=max(5, ROUNDS), lr=LR, val_loader=None, record_name='C. Centralized')
    central_weights = get_weights(central_model)
    with open(MODELS_DIR/'patchtst_centralized.pkl','wb') as f: pickle.dump(central_weights,f)
else:
    with open(MODELS_DIR/'patchtst_centralized.pkl','rb') as f: central_weights = pickle.load(f)


In [ ]:
# Evaluate all strategies on test splits
def eval_strategy(weights, name):
    rows = []
    for bid in usable_bids:
        _, _, te = loaders_for_bid(bid)
        m = make_model(); set_weights(m, weights)
        mae, rmse, cvrmse, wape = eval_model(m, te)
        rows.append({'Strategy':name, 'building_id':bid, 'cold_start': bid in cold_bids, 'mae':mae, 'rmse':rmse, 'cvrmse':cvrmse, 'wape':wape})
    return pd.DataFrame(rows)

results = []

# 0. Local-only
local_rows = []
for bid in usable_bids:
    _, _, te = loaders_for_bid(bid)
    m = make_model(); set_weights(m, local_only_weights[bid])
    mae, rmse, cvrmse, wape = eval_model(m, te)
    local_rows.append({'Strategy':'0. Local-only', 'building_id':bid, 'cold_start': bid in cold_bids, 'mae':mae, 'rmse':rmse, 'cvrmse':cvrmse, 'wape':wape})
results.append(pd.DataFrame(local_rows))

# C. Centralized
results.append(eval_strategy(central_weights, 'C. Centralized'))

results.append(eval_strategy(ftl_weights, 'FTL'))
results.append(eval_strategy(weights_prog, 'Progressive Unfreezing'))
results.append(eval_strategy(weights_inst, 'Instance-TL'))
results.append(eval_strategy(weights_sim, 'Fed-SimTL'))
results.append(eval_strategy(weights_meta, 'FedMetaTL'))

# Personalized-FL evaluation
pers_rows = []
for bid in usable_bids:
    _, _, te = loaders_for_bid(bid)
    if bid in pers_heads:
        w = merge_encoder_head(global_enc, pers_heads[bid])
    else:
        w = merge_encoder_head(global_enc, list(pers_heads.values())[0])
    m = make_model(); set_weights(m, w)
    mae, rmse, cvrmse, wape = eval_model(m, te)
    pers_rows.append({'Strategy':'Personalized-FL', 'building_id':bid, 'cold_start': bid in cold_bids, 'mae':mae, 'rmse':rmse, 'cvrmse':cvrmse, 'wape':wape})
results.append(pd.DataFrame(pers_rows))

all_results = pd.concat(results, ignore_index=True)
all_results.to_csv(RESULTS_DIR / 'patchtst_all6_per_building_metrics.csv', index=False)
display(all_results.head())


In [ ]:
# Summary table: overall / warm / cold
summary_rows = []
for s, d in all_results.groupby('Strategy'):
    overall = d[['mae','rmse','cvrmse','wape']].mean()
    warm = d.loc[~d['cold_start'], ['mae','rmse','cvrmse','wape']].mean()
    cold = d.loc[d['cold_start'], ['mae','rmse','cvrmse','wape']].mean()
    summary_rows.append({'Strategy':s,'Split':'Overall', **overall.to_dict()})
    summary_rows.append({'Strategy':s,'Split':'Warm-start', **warm.to_dict()})
    summary_rows.append({'Strategy':s,'Split':'Cold-start', **cold.to_dict()})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / 'patchtst_all6_summary_metrics.csv', index=False)
display(summary_df)
print('Saved:', RESULTS_DIR / 'patchtst_all6_summary_metrics.csv')


In [ ]:
# Loss-curve figure for PatchTST (all strategies (including baselines))
order = ['0. Local-only','C. Centralized','FTL','Personalized-FL','Progressive Unfreezing','Instance-TL','Fed-SimTL','FedMetaTL']
colors = {
    '0. Local-only':'#17becf',
    'C. Centralized':'#111111',
    'FTL':'#1f77b4',
    'Personalized-FL':'#ff7f0e',
    'Progressive Unfreezing':'#2ca02c',
    'Instance-TL':'#d62728',
    'Fed-SimTL':'#9467bd',
    'FedMetaTL':'#8c564b'
}

grouped = {}
for e in PATCHTST_LOSS_CURVES:
    s = e.get('strategy','Unknown')
    if s == 'meta-inner':
        continue
    grouped.setdefault(s, []).append(np.asarray(e.get('train_loss',[]), dtype=float))

plt.style.use('ggplot')
fig, ax = plt.subplots(figsize=(11,6))
for s in [x for x in order if x in grouped]:
    curves = [c for c in grouped[s] if len(c)>0]
    if not curves:
        continue
    mlen = max(len(c) for c in curves)
    mat = np.vstack([np.pad(c, (0, mlen-len(c)), mode='edge') for c in curves])
    mean = mat.mean(axis=0)
    std = mat.std(axis=0) * 0.15
    x = np.arange(1, len(mean)+1)
    ax.plot(x, mean, lw=2.2, color=colors[s], label=s)
    ax.fill_between(x, mean-std, mean+std, color=colors[s], alpha=0.12)

ax.set_title('PatchTST Training Loss Curves', fontsize=15, pad=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss (MSE)')
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(loc='upper center', bbox_to_anchor=(0.5,-0.14), ncol=3, frameon=False)
plt.tight_layout()
plt.savefig('patchtst_loss_curves_all8.png', dpi=300, bbox_inches='tight')
plt.show()
